In [12]:
import pandas as pd

df = pd.read_csv("/content/DataCo_Cleaned_Interim.csv")

print("Dataset rows:", len(df))
print("Unique orders:", df["Order Id"].nunique())


Dataset rows: 180519
Unique orders: 65752


In [13]:
# 2. Create one record per order
orders = df.groupby("Order Id", as_index=False).agg(
    order_status=("Order Status", "first"),
    delivery_status=("Delivery Status", "first")
)

In [14]:
# 3. Calculate On-Time Delivery Rate
eligible_delivery_statuses = [
    "Advance shipping",
    "Shipping on time",
    "Late delivery"
]

on_time_statuses = [
    "Advance shipping",
    "Shipping on time"
]

eligible_deliveries = orders[
    orders["delivery_status"].isin(
        eligible_delivery_statuses
    )
]

on_time_orders = eligible_deliveries[
    eligible_deliveries["delivery_status"].isin(
        on_time_statuses
    )
]

delivery_numerator = len(on_time_orders)
delivery_denominator = len(eligible_deliveries)

on_time_rate = (
    delivery_numerator / delivery_denominator * 100
    if delivery_denominator else 0
)


In [15]:
# 4. Calculate Cancellation Rate
cancelled_orders = orders[
    orders["order_status"] == "CANCELED"
]

cancellation_numerator = len(cancelled_orders)
cancellation_denominator = len(orders)

cancellation_rate = (
    cancellation_numerator /
    cancellation_denominator * 100
    if cancellation_denominator else 0
)


In [16]:
# 5. Display KPI results
results = pd.DataFrame({
    "KPI": [
        "On-Time Delivery Rate",
        "Cancellation Rate"
    ],
    "Numerator": [
        delivery_numerator,
        cancellation_numerator
    ],
    "Denominator": [
        delivery_denominator,
        cancellation_denominator
    ],
    "Percentage": [
        round(on_time_rate, 2),
        round(cancellation_rate, 2)
    ]
})

print(results.to_string(index=False))


                  KPI  Numerator  Denominator  Percentage
On-Time Delivery Rate      26849        62897       42.69
    Cancellation Rate       1367        65752        2.08


In [17]:
# 6. Validate order counts
print("\nVALIDATION")
print("Raw dataset rows:", len(df))
print("Distinct Order IDs:", df["Order Id"].nunique())
print("Eligible delivered orders:", delivery_denominator)
print("On-time/advance orders:", delivery_numerator)
print("Cancelled orders:", cancellation_numerator)
print("Cancellation denominator:", cancellation_denominator)

assert delivery_numerator <= delivery_denominator
assert cancellation_numerator <= cancellation_denominator
assert orders["Order Id"].is_unique


VALIDATION
Raw dataset rows: 180519
Distinct Order IDs: 65752
Eligible delivered orders: 62897
On-time/advance orders: 26849
Cancelled orders: 1367
Cancellation denominator: 65752


In [18]:
# 7. Export the KPI results
results.to_csv(
    "delivery_Cancellation_KPI results.csv",
    index=False
)
print("\nKPI results exported successfully.")



KPI results exported successfully.
